# Lecture 4 · Manual backprop for $L=(wx+b-y)^2$

**One idea:** *downstream = upstream × local*. We hand-compute the forward pass,
then flow gradients backward through each primitive node and check against
symbolic differentiation and central finite differences. Symbolic differentiation keeps the variables as symbols and derives a new formula; it can be done by hand or by a computer-algebra system. Values: $x=3,\;w=2,\;b=1,\;y=10$.

**Evidence note:** these values are a constructed teaching example. The forward and backward values are exact calculations; finite differences are an independent numerical diagnostic.

In [1]:
# forward pass through primitive ops:  m=wx, a=m+b, e=a-y, L=e^2
x, w, b, y = 3.0, 2.0, 1.0, 10.0
m = w*x
a = m + b
e = a - y
L = e**2
print(f"m={m}, a={a}, e={e}, L={L}")     # expect 6, 7, -3, 9
assert (m, a, e, L) == (6.0, 7.0, -3.0, 9.0)

m=6.0, a=7.0, e=-3.0, L=9.0


## Backward: downstream = upstream × local
Start with $\bar L = 1$ and apply each node's local gradient.

In [2]:
gL = 1.0
ge = gL * (2*e)          # L = e^2  -> dL/de = 2e
ga = ge * 1.0            # e = a - y
gy = ge * (-1.0)
gm = ga * 1.0            # a = m + b
gb = ga * 1.0
gw = gm * x              # m = w*x  -> dm/dw = x
gx = gm * w              #          -> dm/dx = w
print(f"grad w = {gw}")  # -18
print(f"grad b = {gb}")  # -6
print(f"grad x = {gx}")  # -12
print(f"grad y = {gy}")  #  6
assert (gw, gb, gx, gy) == (-18.0, -6.0, -12.0, 6.0)

grad w = -18.0
grad b = -6.0
grad x = -12.0
grad y = 6.0


## Symbolic differentiation: derive a formula first
Keep $w,x,b,y$ symbolic and transform the composed expression with calculus rules. Only after obtaining the derivative formulas do we substitute the numbers. This formula-to-formula route should agree with the numeric local reverse sweep.

In [3]:
residual = w*x + b - y
direct = {
    "w": 2*residual*x,
    "b": 2*residual,
    "x": 2*residual*w,
    "y": -2*residual,
}
manual = {"w": gw, "b": gb, "x": gx, "y": gy}
for name in manual:
    print(f"dL/d{name}: direct={direct[name]:.1f}, backprop={manual[name]:.1f}")
    assert direct[name] == manual[name]

dL/dw: direct=-18.0, backprop=-18.0
dL/db: direct=-6.0, backprop=-6.0
dL/dx: direct=-12.0, backprop=-12.0
dL/dy: direct=6.0, backprop=6.0


## Independent numerical check: central differences

For each scalar $q$, compute $[L(q+\epsilon)-L(q-\epsilon)]/(2\epsilon)$. This is approximate and costs two forward passes per checked coordinate, so it is a diagnostic rather than a training method.

In [4]:
from math import isclose

def loss_value(w_, b_, x_, y_):
    return (w_ * x_ + b_ - y_)**2

eps = 1e-5
values = {"w": w, "b": b, "x": x, "y": y}
finite_difference = {}
for name in values:
    plus = values.copy()
    minus = values.copy()
    plus[name] += eps
    minus[name] -= eps
    estimate = (loss_value(**{f"{k}_": v for k, v in plus.items()})
                - loss_value(**{f"{k}_": v for k, v in minus.items()})) / (2 * eps)
    finite_difference[name] = estimate
    print(f"dL/d{name}: finite difference={estimate: .8f}, backprop={manual[name]: .8f}")
    assert isclose(estimate, manual[name], rel_tol=1e-8, abs_tol=1e-8)

dL/dw: finite difference=-18.00000000, backprop=-18.00000000
dL/db: finite difference=-6.00000000, backprop=-6.00000000
dL/dx: finite difference=-12.00000000, backprop=-12.00000000
dL/dy: finite difference= 6.00000000, backprop= 6.00000000


## One loss-decreasing gradient step ($\eta=0.01$)

In [5]:
eta = 0.01
w_new = w - eta*gw
b_new = b - eta*gb
prediction_new = w_new*x + b_new
loss_new = (prediction_new - y)**2
print(f"w: {w:.2f} -> {w_new:.2f}")
print(f"b: {b:.2f} -> {b_new:.2f}")
print(f"prediction: {a:.2f} -> {prediction_new:.2f} (target {y:.0f})")
print(f"loss: {L:.2f} -> {loss_new:.2f}")
assert isclose(prediction_new, 7.60, abs_tol=1e-12)
assert isclose(loss_new, 5.76, abs_tol=1e-12)
assert loss_new < L

w: 2.00 -> 2.18
b: 1.00 -> 1.06
prediction: 7.00 -> 7.60 (target 10)
loss: 9.00 -> 5.76


## Takeaway
- Backprop is a reverse sweep that repeatedly applies **downstream = upstream × local**, right to left.
- Its derivative values agree with the symbolic formulas at this point ($\nabla_w=-18,\ \nabla_b=-6,\ \nabla_x=-12,\ \nabla_y=6$).
- Central differences agree numerically and provide an independent debugging route.
- The gradient gives the **direction**; the learning rate controls **how far**. Here $\eta=0.01$ lowers the constructed loss from $9$ to $5.76$.